In [1]:
import random
import importlib
import bidders
importlib.reload(bidders)
print("LLMBidder exists:", hasattr(bidders, "LLMBidder"))

from bidders import (
    PositiveMarginBidder,
    BudgetPacedMarginBidder,
    TopKSpecialistBidder,
    LLMBidder,
)
from env import Item, AuctionEnvironment
from scoring import compute_score


# Reproducible demo run
random.seed(7)

# 1) Build 5 items
items = [
    Item("item_1", value=12, rank=1),
    Item("item_2", value=9, rank=2),
    Item("item_3", value=15, rank=3),
    Item("item_4", value=7, rank=4),
    Item("item_5", value=11, rank=5),
]

# 2) Build 4 bidders from bidders.py (includes one LLM bidder)
bidders = [
    PositiveMarginBidder(bidder_id=1, budget=25, beta=1.0, min_bid=0.1),
    BudgetPacedMarginBidder(bidder_id=2, budget=25, beta=1.0, c=1.2, top_k=2),
    TopKSpecialistBidder(bidder_id=3, budget=25, beta=1.0, top_k=3, margin=0.0),
    LLMBidder(bidder_id=4, budget=25, beta=1.0, temperature=0.1, debug=True),
]

# One BidderState per bidder for the episode
states = {b.bidder_id: b.new_state() for b in bidders}

# Simple environment for item ordering/reset
env = AuctionEnvironment(num_agents=len(bidders), items=items)
env.reset()

print("=== AUCTION START ===")
print("Item order:", [f"{it.name}(v={it.value}, r={it.rank})" for it in env.item_order])
print("Bidders:")
for b in bidders:
    st = states[b.bidder_id]
    print(f"  bidder_{b.bidder_id}: {b.__class__.__name__} | budget={st.remaining_budget:.2f}")

MIN_INCREMENT = 1.0

for round_idx, item in enumerate(env.item_order, start=1):
    print("\n" + "=" * 70)
    print(f"ROUND {round_idx} | Auctioning {item.name} (value={item.value}, rank={item.rank})")

    reserve_price = max(1.0, round(item.value * 0.2, 2))
    current_price = reserve_price
    current_winner = None

    active_ids = [b.bidder_id for b in bidders if states[b.bidder_id].remaining_budget >= reserve_price]
    turn_idx = 0

    print(f"Reserve price: {reserve_price:.2f}")
    print(f"Starting active bidders: {active_ids}")

    while len(active_ids) > 1:
        if turn_idx >= len(active_ids):
            turn_idx = 0

        bidder_id = active_ids[turn_idx]
        bidder = next(b for b in bidders if b.bidder_id == bidder_id)
        st = states[bidder_id]

        # If bidder is already leading, skip their turn
        if current_winner == bidder_id:
            turn_idx += 1
            continue

        min_required = current_price + (0.0 if current_winner is None else MIN_INCREMENT)
        if isinstance(bidder, LLMBidder):
            proposed = bidder.place_bid(
                item=item,
                state=st,
                items_remaining=len(env.item_order) - round_idx + 1,
                value_mode="linear",
                current_price=current_price,
                min_required=min_required,
                reserve_price=reserve_price,
                has_current_winner=(current_winner is not None),
            )
        else:
            proposed = bidder.place_bid(
                item=item,
                state=st,
                items_remaining=len(env.item_order) - round_idx + 1,
                value_mode="linear",
            )

        print(
            f"bidder_{bidder_id} ({bidder.__class__.__name__}) | "
            f"budget={st.remaining_budget:.2f} | proposed={proposed:.2f} | "
            f"min_required={min_required:.2f}"
        )

        # Not enough / no bid -> drop out for this item
        if proposed < min_required:
            print(f"  -> bidder_{bidder_id} drops out")
            active_ids.pop(turn_idx)
            continue

        # Cap by remaining budget
        accepted = min(float(proposed), st.remaining_budget)
        if accepted < min_required:
            print(f"  -> bidder_{bidder_id} cannot afford min bid, drops out")
            active_ids.pop(turn_idx)
            continue

        # Accept bid
        current_price = round(accepted, 2)
        current_winner = bidder_id
        item.bids.append(current_price)
        print(f"  -> accepted bid {current_price:.2f} by bidder_{bidder_id}")
        turn_idx += 1

    # Resolve winner
    winner_id = current_winner
    if winner_id is None and len(active_ids) == 1:
        lone_id = active_ids[0]
        lone_state = states[lone_id]
        if lone_state.remaining_budget >= reserve_price:
            winner_id = lone_id
            current_price = reserve_price
            item.bids.append(current_price)

    if winner_id is None:
        print(f"Result: {item.name} UNSOLD")
        continue

    winner_state = states[winner_id]
    winner_state.remaining_budget -= current_price
    winner_state.spent += current_price
    winner_state.items_won.append(item)
    winner_state.prices_paid.append(current_price)

    # Basic utility-like score (value - price)
    delta_score = compute_score(value=item.value, price=current_price, mode="basic", beta=1.0)
    winner_state.total_score += delta_score

    print(
        f"Result: bidder_{winner_id} wins {item.name} for {current_price:.2f} | "
        f"delta_score={delta_score:.2f}"
    )

    # Round budget snapshot
    print("Budgets after round:")
    for b in bidders:
        st = states[b.bidder_id]
        print(f"  bidder_{b.bidder_id}: remaining={st.remaining_budget:.2f}, spent={st.spent:.2f}")

print("\n" + "=" * 70)
print("=== FINAL SUMMARY ===")
for b in bidders:
    st = states[b.bidder_id]
    won_names = [it.name for it in st.items_won]
    print(
        f"bidder_{b.bidder_id} ({b.__class__.__name__}) | "
        f"won={len(won_names)} items {won_names if won_names else '[]'} | "
        f"spent={st.spent:.2f} | remaining={st.remaining_budget:.2f} | "
        f"total_score={st.total_score:.2f}"
    )

print("=== AUCTION END ===")


LLMBidder exists: True
=== AUCTION START ===
Item order: ['item_2(v=9, r=2)', 'item_3(v=15, r=3)', 'item_5(v=11, r=5)', 'item_4(v=7, r=4)', 'item_1(v=12, r=1)']
Bidders:
  bidder_1: PositiveMarginBidder | budget=25.00
  bidder_2: BudgetPacedMarginBidder | budget=25.00
  bidder_3: TopKSpecialistBidder | budget=25.00
  bidder_4: LLMBidder | budget=25.00

ROUND 1 | Auctioning item_2 (value=9, rank=2)
Reserve price: 1.80
Starting active bidders: [1, 2, 3, 4]
bidder_1 (PositiveMarginBidder) | budget=25.00 | proposed=2.98 | min_required=1.80
  -> accepted bid 2.98 by bidder_1
bidder_2 (BudgetPacedMarginBidder) | budget=25.00 | proposed=6.71 | min_required=3.98
  -> accepted bid 6.71 by bidder_2
bidder_3 (TopKSpecialistBidder) | budget=25.00 | proposed=8.21 | min_required=7.71
  -> accepted bid 8.21 by bidder_3
bidder_4 (LLMBidder) | budget=25.00 | proposed=8.22 | min_required=9.21
  -> bidder_4 drops out
bidder_1 (PositiveMarginBidder) | budget=25.00 | proposed=0.74 | min_required=9.21
  -> 